In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle

In [3]:
# Load the dataset

data = pd.read_csv("../Data/processed/final_combined_movie_metadata.csv")
data.head()

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title,comb
0,James Cameron,CCH Pounder,Joel David Moore,Wes Studi,Action Adventure Fantasy Sci-Fi,avatar,CCH Pounder Joel David Moore Wes Studi James C...
1,Gore Verbinski,Johnny Depp,Orlando Bloom,Jack Davenport,Action Adventure Fantasy,pirates of the caribbean: at world's end,Johnny Depp Orlando Bloom Jack Davenport Gore ...
2,Sam Mendes,Christoph Waltz,Rory Kinnear,Stephanie Sigman,Action Adventure Thriller,spectre,Christoph Waltz Rory Kinnear Stephanie Sigman ...
3,Christopher Nolan,Tom Hardy,Christian Bale,Joseph Gordon-Levitt,Action Thriller,the dark knight rises,Tom Hardy Christian Bale Joseph Gordon-Levitt ...
4,Doug Walker,Doug Walker,Rob Walker,unknown,Documentary,star wars: episode vii - the force awakens ...,Doug Walker Rob Walker unknown Doug Walker Doc...


In [ ]:
data['comb'].isnull().sum()

np.int64(0)

In [ ]:
# Convert text into numerical vectors using TF-IDF

tfidf = TfidfVectorizer(stop_words = "english")

tfidf_matrix = tfidf.fit_transform(data['comb'])

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

TF-IDF Matrix Shape: (6145, 10674)


In [7]:
# Compute cosine similarity 

similarity_matrix = cosine_similarity(tfidf_matrix, tfidf_matrix)

print("Similarity Matrix Shape:", similarity_matrix.shape)

Similarity Matrix Shape: (6145, 6145)


In [16]:
def recommed_movie(movie_title, similarity = similarity_matrix, top_n = 10):
    movie_title = movie_title.lower()

    if movie_title not in data["movie_title"].str.lower().values:
        return "Movie not found in the dataset."
    

    idx = data[data["movie_title"].str.lower() == movie_title].index[0]

    # getting similarity scores for the requested movie with all other movies
    similarity_scores = list(enumerate(similarity[idx]))
    
    # Sorting the movie based on similarity scores (highest first)
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

    # getting top n similar movies excluding the first one which is the requested movie itself
    top_movies = similarity_scores[1:top_n+1] 

    recommeded_movies = [data["movie_title"].iloc[i[0]] for i in top_movies]
    return recommeded_movies

In [18]:
print(recommed_movie("Inception"))

['the dark knight rises', 'don jon', 'the revenant', 'g.i. joe: the rise of cobra', '3rd rock from the sun            ', 'treasure planet', 'sin city: a dame to kill for', 'looper', '7500', 'catch me if you can']


In [19]:
import pickle

# Save TF-IDF vectorizer
pickle.dump(tfidf, open("../model/tfidf_vectorizer.pkl", "wb"))

# save similarity matrix
pickle.dump(similarity_matrix, open("../model/similarity_matrix.pkl", "wb"))